# Manual Bronze-to-Silver backfill

This notebook uses the same data-access, transformation, and loading boundaries as the Silver command. Completed runs are skipped automatically, while errors stop the backfill immediately.

In [ ]:
import logging

import pandas as pd
from IPython.display import display

from pipeline import (
    extract_pending_canonical_bronze,
    get_database_connection,
    init_motherduck_config,
    load_silver,
    transform_extraction_to_silver,
)
from pipeline.etl import create_pipeline_tables

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("manual_silver_backfill")

## Load pending runs

The data-access layer selects canonical pending runs and converts their stored JSON payloads into typed `Extraction` objects. Only non-payload metadata is previewed.

In [ ]:
motherduck_config = init_motherduck_config()

with get_database_connection(motherduck_config) as conn:
    create_pipeline_tables(conn)
    pending_extractions = extract_pending_canonical_bronze(conn)
    pending_metadata = pd.DataFrame(
        [
            {
                "run_id": extraction.run_id,
                "attempt_number": extraction.attempt_number,
                "extracted_at": extraction.extracted_at,
            }
            for extraction in pending_extractions
        ],
        columns=["run_id", "attempt_number", "extracted_at"],
    )

    print(f"Pending Bronze runs: {len(pending_extractions)}")
    display(pending_metadata)

    for extraction in pending_extractions:
        silver_df = transform_extraction_to_silver(extraction)
        load_silver(conn, silver_df)
        logger.info(
            "Loaded Silver run_id=%s attempt_number=%s rows=%s",
            extraction.run_id,
            extraction.attempt_number,
            len(silver_df),
        )

    pending_after_load = len(extract_pending_canonical_bronze(conn))
    silver_rows_by_run = conn.execute(
        """
        SELECT run_id, count(*) AS silver_rows
        FROM silver.issue_submissions
        GROUP BY run_id
        ORDER BY run_id
        """
    ).fetchdf()
    primary_key_duplicates = conn.execute(
        """
        SELECT run_id, issue_id, count(*) AS duplicate_count
        FROM silver.issue_submissions
        GROUP BY run_id, issue_id
        HAVING count(*) > 1
        ORDER BY run_id, issue_id
        """
    ).fetchdf()
    builder_milestone_duplicates = conn.execute(
        """
        SELECT
            run_id,
            issue_author,
            milestone,
            count(*) AS duplicate_count
        FROM silver.issue_submissions
        GROUP BY run_id, issue_author, milestone
        HAVING count(*) > 1
        ORDER BY run_id, issue_author, milestone
        """
    ).fetchdf()

## Verification

In [ ]:
print(f"Pending Bronze runs after load: {pending_after_load}")
print(f"Duplicate (run_id, issue_id) keys: {len(primary_key_duplicates)}")
print(
    "Duplicate (run_id, issue_author, milestone) keys: "
    f"{len(builder_milestone_duplicates)}"
)
display(silver_rows_by_run)
display(primary_key_duplicates)
display(builder_milestone_duplicates)